# 12-02 TabPFN Top10 Model Selection und Tuning

Dieses Notebook testet normale `TabPFNClassifier`-Einstellungen auf `Top10` als vollstaendiges Parameter-Grid.

Wichtig:
- Training: `X_train` + `y_top10_train` (`<=2022`).
- Validierung: `X_valid` + `y_top10_valid` (`2023`).
- Auswahlmetrik: `roc_auc`.
- Das Tuning laeuft als Cartesian Product ueber alle unterstuetzten erlaubten Parameter.
- `thinking_mode`, `thinking_effort`, `thinking_metric` sind in diesem Notebook verboten.
- Durchlaufzeiten werden pro Run erhoben.
- Fuer `tabpfn_client` ist `n_estimators` in der aktuellen API serverseitig auf maximal `8` begrenzt.
- `RUN_API = False` ist der sichere Default.


In [23]:
from itertools import product
from pathlib import Path
from time import perf_counter
import json
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "data" / "model_data").exists() and (candidate / "src" / "Notebooks").exists():
            return candidate
    raise FileNotFoundError("Projektwurzel mit data/model_data und src/Notebooks nicht gefunden.")

PROJECT_ROOT = find_project_root()
MODEL_DATA_DIR = PROJECT_ROOT / "data" / "model_data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABPFN_DIR = PROCESSED_DIR / "tabpfn"
LEGACY_DIR = PROCESSED_DIR / "tabpfn_3_experiments"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"MODEL_DATA_DIR: {MODEL_DATA_DIR}")


PROJECT_ROOT: /Users/roberthendrich/GADA-Group3-Cycling-Stage-Prediction
MODEL_DATA_DIR: /Users/roberthendrich/GADA-Group3-Cycling-Stage-Prediction/data/model_data


In [24]:
FEATURE_COLUMNS = [
    "distance",
    "vertical_meters",
    "stage_nr",
    "team_tier",
    "age_at_race",
    "rider_bmi",
    "wind_stability_index",
    "weather_temp_mean",
    "weather_temp_trend",
    "weather_rain_prob_mean",
    "weather_precipitation_mean",
    "weather_humidity_mean",
    "gradient_final_km",
    "lag_rider_points_season",
    "lag_rider_rank_season",
    "lag_race_competitiveness_median",
    "lag_team_power_index",
]

TARGET_CONFIGS = {
    "top5": {
        "label": "Top5",
        "train_file": "y_top5_train.pkl",
        "valid_file": "y_top5_valid.pkl",
        "test_file": "y_top5_test.pkl",
        "score_col": "p_top5_raw",
        "actual_col": "actual_top5",
    },
    "top10": {
        "label": "Top10",
        "train_file": "y_top10_train.pkl",
        "valid_file": "y_top10_valid.pkl",
        "test_file": "y_top10_test.pkl",
        "score_col": "p_top10_raw",
        "actual_col": "actual_top10",
    },
    "top20": {
        "label": "Top20",
        "train_file": "y_top20_train.pkl",
        "valid_file": "y_top20_valid.pkl",
        "test_file": "y_top20_test.pkl",
        "score_col": "p_top20_raw",
        "actual_col": "actual_top20",
    },
}

EXPECTED_SHAPES = {
    "train": (169349, 17),
    "valid": (8897, 17),
    "test": (17802, 17),
}


In [25]:
RUN_API = True
FORCE_RERUN = False
SEED = 42
TUNING_TARGET = "top10"
PRIMARY_METRIC = "roc_auc"

# Die aktuelle Prior-Labs-API fuer tabpfn_client lehnt n_estimators > 8 serverseitig ab.
TABPFN_CLIENT_N_ESTIMATORS_MAX = 8
N_ESTIMATORS_GRID = [4, 8]

RESULT_DIR = TABPFN_DIR / "02_tuning_top10"
PREDICTION_DIR = RESULT_DIR / "predictions"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

VALIDATION_FORBIDDEN_PARAMS = {"thinking_mode", "thinking_effort", "thinking_metric"}


def seconds_since(start):
    return round(perf_counter() - start, 3)


def safe_json_dumps(obj):
    return json.dumps(obj, sort_keys=True, ensure_ascii=True, default=str)


def params_short(params):
    if not params:
        return "defaults"
    return "__".join(f"{k}-{v}" for k, v in sorted(params.items()))


def sanitize_label(value):
    keep = []
    for char in str(value):
        keep.append(char if char.isalnum() or char in "-_" else "_")
    return "".join(keep).strip("_")


## TabPFN-Client und unterstuetzte Parameter pruefen

Wenn `tabpfn-client` oder `tabpfn` nicht installiert ist, bleibt das Notebook trotzdem lauffaehig und nutzt vorhandene Caches.


In [26]:
TabPFNClassifier = None
client_import_error = None
try:
    from tabpfn_client import TabPFNClassifier as _TabPFNClassifier
    TabPFNClassifier = _TabPFNClassifier
    client_source = "tabpfn_client"
except Exception as exc_client:
    try:
        from tabpfn import TabPFNClassifier as _TabPFNClassifier
        TabPFNClassifier = _TabPFNClassifier
        client_source = "tabpfn"
    except Exception as exc_local:
        client_import_error = f"tabpfn_client: {exc_client}; tabpfn: {exc_local}"
        client_source = None

if TabPFNClassifier is None:
    warnings.warn(f"Kein TabPFNClassifier importierbar. API-Laeufe bleiben deaktiviert. {client_import_error}")
    supported_params = {}
else:
    try:
        supported_params = TabPFNClassifier().get_params()
    except Exception as exc:
        warnings.warn(f"TabPFNClassifier konnte nicht instanziiert werden: {exc}")
        supported_params = {}

planned_params = {
    "n_estimators": N_ESTIMATORS_GRID,
    "softmax_temperature": [0.6, 0.8, 1.0, 1.2, 1.4],
    "average_before_softmax": [True, False],
    "balance_probabilities": [True, False],
    "thinking_mode": [],
    "thinking_effort": [],
    "thinking_metric": [],
}

supported_rows = []
for param, values in planned_params.items():
    is_supported = param in supported_params
    if param in VALIDATION_FORBIDDEN_PARAMS:
        status = "nur finaler Test in 12-03"
    elif is_supported:
        status = "test im Full Grid in 12-02"
    else:
        status = "nicht unterstuetzt oder unbekannt"
    supported_rows.append({
        "parameter": param,
        "supported_by_classifier": is_supported,
        "planned_values": values,
        "validation_status": status,
        "client_source": client_source,
    })

supported_param_df = pd.DataFrame(supported_rows)
supported_param_df.to_csv(RESULT_DIR / "classifier_supported_params.csv", index=False)
supported_param_df


,parameter,supported_by_classifier,planned_values,validation_status,client_source
0,n_estimators,True,"[4, 8]",test im Full Grid in 12-02,tabpfn_client
1,softmax_temperature,True,"[0.6, 0.8, 1.0, 1.2, 1.4]",test im Full Grid in 12-02,tabpfn_client
2,average_before_softmax,True,"[True, False]",test im Full Grid in 12-02,tabpfn_client
3,balance_probabilities,True,"[True, False]",test im Full Grid in 12-02,tabpfn_client
4,thinking_mode,True,[],nur finaler Test in 12-03,tabpfn_client
5,thinking_effort,True,[],nur finaler Test in 12-03,tabpfn_client
6,thinking_metric,True,[],nur finaler Test in 12-03,tabpfn_client


## Top10-Daten laden


In [27]:
def read_pickle(name):
    path = MODEL_DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_pickle(path)

X_train = read_pickle("X_train.pkl")
X_valid = read_pickle("X_valid.pkl")
y_top10_train = read_pickle("y_top10_train.pkl")
y_top10_valid = read_pickle("y_top10_valid.pkl")
y_rank_valid = read_pickle("y_rank_valid.pkl")
groups_valid = read_pickle("groups_valid.pkl")
meta_valid = read_pickle("meta_valid.pkl")

assert list(X_train.columns) == FEATURE_COLUMNS
assert list(X_valid.columns) == FEATURE_COLUMNS
if X_train.shape != EXPECTED_SHAPES["train"]:
    warnings.warn(f"X_train Shape weicht ab: {X_train.shape}")
if X_valid.shape != EXPECTED_SHAPES["valid"]:
    warnings.warn(f"X_valid Shape weicht ab: {X_valid.shape}")
assert set(meta_valid["meta_year"].unique()) == {2023}
assert pd.Series(y_top10_train).isin([0, 1]).all()
assert pd.Series(y_top10_valid).isin([0, 1]).all()

print("X_train", X_train.shape)
print("X_valid", X_valid.shape)
print("Top10 train positive rate", float(pd.Series(y_top10_train).mean()))
print("Top10 valid positive rate", float(pd.Series(y_top10_valid).mean()))


X_train (169349, 17)
X_valid (8897, 17)
Top10 train positive rate 0.05908508464768023
Top10 valid positive rate 0.06406653928290434


## Run-Helfer


In [28]:
from sklearn.metrics import average_precision_score, roc_auc_score


def filter_supported_validation_params(candidate_params, supported_params):
    return {
        k: v
        for k, v in candidate_params.items()
        if k in supported_params and k not in VALIDATION_FORBIDDEN_PARAMS
    }


def validation_api_limit_error(params):
    if client_source == "tabpfn_client" and "n_estimators" in params:
        try:
            n_estimators = int(params["n_estimators"])
        except (TypeError, ValueError):
            return f"n_estimators ist nicht als int interpretierbar: {params['n_estimators']}"
        if n_estimators > TABPFN_CLIENT_N_ESTIMATORS_MAX:
            return (
                "tabpfn_client/API-Limit: n_estimators muss <= "
                f"{TABPFN_CLIENT_N_ESTIMATORS_MAX} sein, erhalten: {n_estimators}"
            )
    return None


def positive_class_probability(model, X):
    proba = model.predict_proba(X)
    classes = list(model.classes_)
    if 1 not in classes:
        raise ValueError(f"Positive Klasse 1 fehlt in model.classes_: {classes}")
    positive_idx = classes.index(1)
    return proba[:, positive_idx]


def build_validation_prediction_df(p_top10, run_label, params):
    pred = meta_valid.reset_index(drop=True).copy()
    pred["stage_id"] = pd.Series(groups_valid).reset_index(drop=True).astype(str).values
    pred["actual_rank"] = pd.Series(y_rank_valid).reset_index(drop=True).values
    pred["actual_top10"] = pd.Series(y_top10_valid).reset_index(drop=True).astype(int).values
    pred["p_top10_valid"] = p_top10
    pred["run_label"] = run_label
    pred["target"] = TUNING_TARGET
    for key, value in params.items():
        pred[key] = value
    return pred


def run_validation(run_label, candidate_params):
    run_start = perf_counter()
    if VALIDATION_FORBIDDEN_PARAMS.intersection(candidate_params):
        raise ValueError(f"Thinking-Parameter sind in 12-02 verboten: {candidate_params}")

    params = filter_supported_validation_params(candidate_params, supported_params)
    pred_path = PREDICTION_DIR / f"{sanitize_label(run_label)}.csv"
    runtime = {
        "run_label": run_label,
        "target": TUNING_TARGET,
        "params_json": safe_json_dumps(params),
        "params_short": params_short(params),
        "rows_train": len(X_train),
        "rows_valid": len(X_valid),
        "fit_seconds": np.nan,
        "predict_seconds": np.nan,
        "total_seconds": np.nan,
        "cache_load_seconds": np.nan,
        "runtime_status": None,
        "error_message": None,
    }

    limit_error = validation_api_limit_error(params)
    if limit_error is not None:
        runtime["runtime_status"] = "skipped_api_limit"
        runtime["error_message"] = limit_error
        runtime["total_seconds"] = seconds_since(run_start)
        return None, runtime, {
            "run_label": run_label,
            "target": TUNING_TARGET,
            "params_json": safe_json_dumps(params),
            "params_short": params_short(params),
            "roc_auc": np.nan,
            "average_precision": np.nan,
            "prediction_path": str(pred_path),
            "status": "skipped_api_limit",
            "error_message": limit_error,
        }

    pred = None
    if pred_path.exists() and not FORCE_RERUN:
        cache_start = perf_counter()
        pred = pd.read_csv(pred_path)
        runtime["cache_load_seconds"] = seconds_since(cache_start)
        runtime["runtime_status"] = "cache_hit"
    elif not RUN_API:
        runtime["runtime_status"] = "skipped_missing_cache"
        runtime["total_seconds"] = seconds_since(run_start)
        return None, runtime, {"run_label": run_label, "target": TUNING_TARGET, "params_json": safe_json_dumps(params), "params_short": params_short(params), "roc_auc": np.nan, "average_precision": np.nan, "prediction_path": str(pred_path), "status": "skipped_missing_cache", "error_message": None}
    else:
        if TabPFNClassifier is None:
            runtime["runtime_status"] = "missing_classifier"
            runtime["total_seconds"] = seconds_since(run_start)
            return None, runtime, {"run_label": run_label, "target": TUNING_TARGET, "params_json": safe_json_dumps(params), "params_short": params_short(params), "roc_auc": np.nan, "average_precision": np.nan, "prediction_path": str(pred_path), "status": "missing_classifier", "error_message": client_import_error}

        try:
            fit_start = perf_counter()
            model = TabPFNClassifier(**params)
            model.fit(X_train, y_top10_train)
            runtime["fit_seconds"] = seconds_since(fit_start)

            predict_start = perf_counter()
            p_top10 = positive_class_probability(model, X_valid)
            runtime["predict_seconds"] = seconds_since(predict_start)

            pred = build_validation_prediction_df(p_top10, run_label, params)
            pred.to_csv(pred_path, index=False)
            runtime["runtime_status"] = "api_run"
        except Exception as exc:
            runtime["runtime_status"] = "api_error"
            runtime["error_message"] = repr(exc)
            runtime["total_seconds"] = seconds_since(run_start)
            return None, runtime, {
                "run_label": run_label,
                "target": TUNING_TARGET,
                "params_json": safe_json_dumps(params),
                "params_short": params_short(params),
                "prediction_path": str(pred_path),
                "status": "api_error",
                "roc_auc": np.nan,
                "average_precision": np.nan,
                "error_message": repr(exc),
            }

    runtime["total_seconds"] = seconds_since(run_start)

    metrics = {
        "run_label": run_label,
        "target": TUNING_TARGET,
        "params_json": safe_json_dumps(params),
        "params_short": params_short(params),
        "prediction_path": str(pred_path),
        "status": runtime["runtime_status"],
        "roc_auc": np.nan,
        "average_precision": np.nan,
        "error_message": runtime.get("error_message"),
    }
    if pred is not None and "p_top10_valid" in pred.columns:
        y_true = pred["actual_top10"].astype(int)
        y_score = pred["p_top10_valid"].astype(float)
        if y_true.nunique() == 2:
            metrics["roc_auc"] = roc_auc_score(y_true, y_score)
            metrics["average_precision"] = average_precision_score(y_true, y_score)

    return pred, runtime, metrics


## Full-Grid-Kandidaten definieren

Alle unterstuetzten erlaubten Validierungsparameter werden als Cartesian Product kombiniert. Nicht unterstuetzte Parameter und API-Limit-Verletzungen werden dokumentiert und nicht ausgefuehrt.

In [29]:
GRID_PARAM_ORDER = [
    "n_estimators",
    "softmax_temperature",
    "average_before_softmax",
    "balance_probabilities",
]


def build_validation_grid(planned_params, supported_params):
    active_grid = {}
    skipped_grid_rows = []

    for param in GRID_PARAM_ORDER:
        planned_values = list(planned_params.get(param, []))
        if param in VALIDATION_FORBIDDEN_PARAMS:
            skipped_grid_rows.append({
                "parameter": param,
                "reason": "forbidden_in_validation",
                "planned_values": planned_values,
            })
            continue
        if param not in supported_params:
            skipped_grid_rows.append({
                "parameter": param,
                "reason": "not_supported_by_classifier",
                "planned_values": planned_values,
            })
            continue

        valid_values = []
        for value in planned_values:
            candidate = {param: value}
            limit_error = validation_api_limit_error(candidate)
            if limit_error is not None:
                skipped_grid_rows.append({
                    "parameter": param,
                    "reason": "api_limit",
                    "planned_values": [value],
                    "error_message": limit_error,
                })
                continue
            valid_values.append(value)

        if valid_values:
            active_grid[param] = valid_values
        else:
            skipped_grid_rows.append({
                "parameter": param,
                "reason": "no_valid_values_after_filtering",
                "planned_values": planned_values,
            })

    if not active_grid:
        warnings.warn("Keine unterstuetzten Validierungsparameter fuer das Grid gefunden.")
        return [], pd.DataFrame(), pd.DataFrame(skipped_grid_rows)

    grid_keys = list(active_grid.keys())
    raw_combinations = [dict(zip(grid_keys, values)) for values in product(*(active_grid[key] for key in grid_keys))]

    unique_candidate_runs = []
    seen = set()
    for params in raw_combinations:
        filtered = filter_supported_validation_params(params, supported_params)
        limit_error = validation_api_limit_error(filtered)
        if limit_error is not None:
            skipped_grid_rows.append({
                "parameter": "__combination__",
                "reason": "api_limit",
                "planned_values": filtered,
                "error_message": limit_error,
            })
            continue
        key = safe_json_dumps(filtered)
        if key in seen:
            continue
        seen.add(key)
        run_label = f"grid_{len(unique_candidate_runs) + 1:03d}"
        unique_candidate_runs.append((run_label, filtered))

    candidate_grid_df = pd.DataFrame([
        {
            "run_label": label,
            "params_short": params_short(params),
            "params_json": safe_json_dumps(params),
            **params,
        }
        for label, params in unique_candidate_runs
    ])
    skipped_grid_df = pd.DataFrame(skipped_grid_rows)
    return unique_candidate_runs, candidate_grid_df, skipped_grid_df


unique_candidate_runs, candidate_grid_df, skipped_grid_df = build_validation_grid(planned_params, supported_params)

candidate_grid_df.to_csv(RESULT_DIR / "top10_tuning_candidate_grid.csv", index=False)
skipped_grid_df.to_csv(RESULT_DIR / "top10_tuning_grid_skipped_params.csv", index=False)

print(f"Grid-Runs: {len(unique_candidate_runs)}")
candidate_grid_df


Grid-Runs: 40


,run_label,params_short,params_json,n_estimators,softmax_temperature,average_before_softmax,balance_probabilities
0,grid_001,average_before_softmax-True__balance_probabili...,"{""average_before_softmax"": true, ""balance_prob...",4,0.6,True,True
1,grid_002,average_before_softmax-True__balance_probabili...,"{""average_before_softmax"": true, ""balance_prob...",4,0.6,True,False
2,grid_003,average_before_softmax-False__balance_probabil...,"{""average_before_softmax"": false, ""balance_pro...",4,0.6,False,True
3,grid_004,average_before_softmax-False__balance_probabil...,"{""average_before_softmax"": false, ""balance_pro...",4,0.6,False,False
4,grid_005,average_before_softmax-True__balance_probabili...,"{""average_before_softmax"": true, ""balance_prob...",4,0.8,True,True
5,grid_006,average_before_softmax-True__balance_probabili...,"{""average_before_softmax"": true, ""balance_prob...",4,0.8,True,False
6,grid_007,average_before_softmax-False__balance_probabil...,"{""average_before_softmax"": false, ""balance_pro...",4,0.8,False,True
7,grid_008,average_before_softmax-False__balance_probabil...,"{""average_before_softmax"": false, ""balance_pro...",4,0.8,False,False
8,grid_009,average_before_softmax-True__balance_probabili...,"{""average_before_softmax"": true, ""balance_prob...",4,1.0,True,True
9,grid_010,average_before_softmax-True__balance_probabili...,"{""average_before_softmax"": true, ""balance_prob...",4,1.0,True,False


## Validierungslaeufe ausfuehren oder aus Cache laden


In [30]:
runtime_rows = []
metric_rows = []
run_rows = []

for run_label, params in unique_candidate_runs:
    print(f"Run: {run_label} {params}")
    pred, runtime, metrics = run_validation(run_label, params)
    runtime_rows.append(runtime)
    metric_rows.append({**metrics, **{k: runtime.get(k) for k in ["fit_seconds", "predict_seconds", "total_seconds", "cache_load_seconds", "runtime_status", "error_message"]}})
    run_rows.append({
        "run_label": run_label,
        "target": TUNING_TARGET,
        "params_short": params_short(params),
        "params_json": safe_json_dumps(params),
        "status": metrics.get("status"),
    })

runtime_df = pd.DataFrame(runtime_rows)
metrics_df = pd.DataFrame(metric_rows)
runs_df = pd.DataFrame(run_rows)

runtime_df.to_csv(RESULT_DIR / "top10_tuning_runtime.csv", index=False)
metrics_df.to_csv(RESULT_DIR / "top10_tuning_metrics.csv", index=False)
runs_df.to_csv(RESULT_DIR / "top10_tuning_runs.csv", index=False)

# Usage wird als eigene Datei angelegt, auch wenn keine API-Usage abrufbar ist.
usage_df = pd.DataFrame([{"run_label": r["run_label"], "usage_status": "not_captured_in_notebook", "note": "Token/credit usage must be filled from client-specific usage APIs if available."} for r in run_rows])
usage_df.to_csv(RESULT_DIR / "top10_tuning_usage.csv", index=False)

metrics_df


Run: grid_001 {'n_estimators': 4, 'softmax_temperature': 0.6, 'average_before_softmax': True, 'balance_probabilities': True}
00:00 Fitting... \

The provided train set hashes match previously uploaded train sets.


00:00 Fitting... Done!
00:00 Predicting... -

The provided test set hash matches a previously uploaded test set.


00:23 Predicting... Done!
Run: grid_002 {'n_estimators': 4, 'softmax_temperature': 0.6, 'average_before_softmax': True, 'balance_probabilities': False}
00:00 Fitting... -

...

The provided train set hashes match previously uploaded train sets.


00:00 Fitting... Done!
00:00 Predicting... \

The provided test set hash matches a previously uploaded test set.


00:45 Predicting... Done!


,run_label,target,params_json,params_short,prediction_path,status,roc_auc,average_precision,error_message,fit_seconds,predict_seconds,total_seconds,cache_load_seconds,runtime_status
0,grid_001,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.430,23.255,23.723,NaN,api_run
1,grid_002,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.429,23.459,23.924,NaN,api_run
2,grid_003,top10,"{""average_before_softmax"": false, ""balance_pro...",average_before_softmax-False__balance_probabil...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.782625,0.272738,None,0.421,23.309,23.763,NaN,api_run
3,grid_004,top10,"{""average_before_softmax"": false, ""balance_pro...",average_before_softmax-False__balance_probabil...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.782625,0.272738,None,0.425,23.465,23.923,NaN,api_run
4,grid_005,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.426,23.265,23.728,NaN,api_run
5,grid_006,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.428,23.269,23.730,NaN,api_run
6,grid_007,top10,"{""average_before_softmax"": false, ""balance_pro...",average_before_softmax-False__balance_probabil...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.782812,0.273525,None,0.423,23.280,23.738,NaN,api_run
7,grid_008,top10,"{""average_before_softmax"": false, ""balance_pro...",average_before_softmax-False__balance_probabil...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.782812,0.273525,None,0.425,23.086,23.544,NaN,api_run
8,grid_009,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.434,23.027,23.496,NaN,api_run
9,grid_010,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.430,23.479,23.944,NaN,api_run


## Runtime-Vergleich und finale Auswahl

Primaer entscheidet `roc_auc`. Laufzeit ist ein Tie-Breaker.


In [32]:
valid_metrics = metrics_df.dropna(subset=["roc_auc"]).copy() if not metrics_df.empty else pd.DataFrame()

if valid_metrics.empty:
    warnings.warn("Keine gueltigen Metriken vorhanden. Entweder RUN_API=True setzen oder passende Prediction-Caches bereitstellen.")
    decision_table = metrics_df.copy()
else:
    decision_table = valid_metrics.sort_values(
        ["roc_auc", "average_precision", "total_seconds"],
        ascending=[False, False, True],
        na_position="last",
    ).reset_index(drop=True)

decision_table.to_csv(RESULT_DIR / "top10_tuning_decision_table.csv", index=False)

if not valid_metrics.empty:
    best = decision_table.iloc[0].to_dict()
    best_params = json.loads(best["params_json"])
    selected = {
        "model_family": "TabPFNClassifier",
        "selection_target": "top10",
        "selection_metric": "roc_auc",
        "selection_train_context": "X_train, meta_year <= 2022",
        "selection_validation_context": "X_valid, meta_year == 2023",
        "selection_train_rows": int(len(X_train)),
        "validation_rows": int(len(X_valid)),
        "params": best_params,
        "excluded_from_validation": sorted(VALIDATION_FORBIDDEN_PARAMS),
        "validation_metrics": {
            "roc_auc": None if pd.isna(best.get("roc_auc")) else float(best.get("roc_auc")),
            "average_precision": None if pd.isna(best.get("average_precision")) else float(best.get("average_precision")),
        },
        "validation_runtime": {
            "fit_seconds": None if pd.isna(best.get("fit_seconds")) else float(best.get("fit_seconds")),
            "predict_seconds": None if pd.isna(best.get("predict_seconds")) else float(best.get("predict_seconds")),
            "total_seconds": None if pd.isna(best.get("total_seconds")) else float(best.get("total_seconds")),
        },
        "selection_reason": "Best validation roc_auc; runtime used as tie-breaker.",
    }
    with open(RESULT_DIR / "tabpfn_selected_classifier_params.json", "w") as f:
        json.dump(selected, f, indent=2, ensure_ascii=True)
else:
    selected = None

decision_table


,run_label,target,params_json,params_short,prediction_path,status,roc_auc,average_precision,error_message,fit_seconds,predict_seconds,total_seconds,cache_load_seconds,runtime_status
0,grid_009,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.434,23.027,23.496,NaN,api_run
1,grid_017,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.426,23.240,23.699,NaN,api_run
2,grid_001,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.430,23.255,23.723,NaN,api_run
3,grid_005,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.426,23.265,23.728,NaN,api_run
4,grid_006,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.428,23.269,23.730,NaN,api_run
5,grid_014,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.447,23.243,23.732,NaN,api_run
6,grid_018,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.427,23.458,23.920,NaN,api_run
7,grid_002,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.429,23.459,23.924,NaN,api_run
8,grid_013,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.425,23.455,23.937,NaN,api_run
9,grid_010,top10,"{""average_before_softmax"": true, ""balance_prob...",average_before_softmax-True__balance_probabili...,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,api_run,0.783128,0.274043,None,0.430,23.479,23.944,NaN,api_run
